# Contagion engine: stress test demo

1. Set a macro shock in natural units.
2. Read the ranked table. The **hybrid engine** gives the ranking: a seasonal ridge base path plus the accounting transmission of the shock to the debt of each company. The **neural engine** (the graph recurrent model) is shown next to it, for comparison.
3. Select a ticker and read its audit trail.

Before you run this notebook, build the data and train the models (see `README.md`).

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
root = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(root); sys.path.insert(0, root)
import pandas as pd
from src.simulator.stress_test import simulate
from src.knowledge.audit import Auditor
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
pd.set_option('display.width', 200)

## 1. How good is each engine?

1-step forecast error on the test split (2022-Q1 to 2024-Q4), divided by the usual quarterly change of each company. Lower is better. `mean_change` is the naive forecast that both engines start from.

In [ ]:
metrics = json.load(open('results/metrics.json'))
rows = {f'baseline {k}': v for k, v in metrics['baselines'].items()}
rows['hybrid ridge'] = metrics['ridge']
rows.update({f'neural {k}': v for k, v in metrics['models'].items()})
pd.DataFrame(rows).T[['mse_std', 'mae_std', 'median_abs_std', 'train_mse_std']]

## 2. The shock

- `SELIC_delta`: percentage points of the SELIC target. `3.0` is +300 basis points.
- `IPCA_delta`: percentage points of the quarterly IPCA change.
- `PTAX_delta_pct`: percent change of the USD/BRL rate. `15` is +15 %.

The shock starts after the last filed quarter and holds for 4 quarters. `hybrid_change_pct` and `neural_change_pct` are the equity change caused by the shock, as a percent of the last filed equity.

In [ ]:
shocks = {'SELIC_delta': 3.0, 'IPCA_delta': 0.0, 'PTAX_delta_pct': 15.0}
scenario = simulate(shocks, horizon=4)
table = scenario.table()
columns = ['rank', 'ticker', 'sector', 'equity_filed', 'hybrid_change_pct', 'neural_change_pct', 'neural_rank',
           'floating_share', 'floating_basis', 'coverage_under_shock', 'breaching_suppliers']
table[columns].head(15)

## 3. Checks

Each predicted balance sheet must balance. A zero shock must give a zero change in both engines.

In [ ]:
print('rank correlation, hybrid vs neural:', round(scenario.rank_correlation(), 2))
print('largest balance residual / assets:', f"hybrid {table['hybrid_residual'].max():.2e}, neural {table['neural_residual'].max():.2e}")
zero = simulate({}, horizon=4).table()
print('largest change with a zero shock (%):', float(zero['hybrid_change_pct'].abs().max()), float(zero['neural_change_pct'].abs().max()))

## 4. The audit trail of 1 company

Each line says where its numbers come from: `input`, `filed`, `extracted`, `rule`, `model`, `estimate` or `none`.

In [ ]:
auditor = Auditor(scenario)
ticker = table.loc[~table['equity_not_positive']].iloc[0]['ticker']  # change this to any ticker of the table
print(auditor.render(ticker))

In [ ]:
print(auditor.render('PETR4'))

## Limits

- The hybrid shock effect is an accounting pass-through with stated assumptions (34 % tax, no change in revenue or costs). It is not learned from data.
- The neural shock response is an extrapolation from 45 quarters of macro data. The neural engine does not beat the naive forecast on the test split.
- The universe holds companies with all quarters from 2010-Q4 to 2024-Q4 (survivors). Statements are the latest filed version, not point-in-time.
- The extraction covers only what a company discloses. An attention weight is not proof of a causal link.